# Starter Notebook - MLflow Basics

This notebook demonstrates fundamental MLflow concepts:
- `mlflow.autolog()` for automatic experiment tracking
- Logging a model manually
- Loading a saved model and making predictions

**Dataset:** California Housing (sklearn built-in)  
**Model:** Decision Tree Regressor  
**Task:** Predict median house value based on housing features

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
mlflow.set_tracking_uri("file:./mlruns")

## 1. Load and Explore the Dataset

In [2]:
# Load California Housing dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"\nFirst 5 rows:")
X.head()

Dataset shape: (20640, 8)
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

First 5 rows:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 16512, Test size: 4128


## 2. MLflow Autolog

With `mlflow.autolog()`, MLflow automatically logs parameters, metrics, and the model artifact without any manual logging calls.

In [4]:
# Enable autologging
mlflow.autolog()

# Set experiment name
mlflow.set_experiment("california-housing-starter")

# Train a Decision Tree model - autolog captures everything
with mlflow.start_run(run_name="autolog-decision-tree"):
    dt_model = DecisionTreeRegressor(max_depth=10, min_samples_split=5, random_state=42)
    dt_model.fit(X_train, y_train)
    
    # Predictions
    y_pred = dt_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"Autolog Run - MSE: {mse:.4f}, R2: {r2:.4f}")

# Disable autolog for subsequent manual logging
mlflow.autolog(disable=True)

2026/03/14 22:03:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
/Users/harsha/Desktop/MLOps Labs/MLOps_Labs/lab4/mlflow_lab1/mlflow_lab1_env/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/14 22:03:06 INFO mlflow.tracking.fluent: Experiment with name 'california-housing-starter' does not exist. Creating a new experiment.
2026/03/14 22:03:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during d

Autolog Run - MSE: 0.4109, R2: 0.6864


## 3. Manual Model Logging

Now we manually log a model with `mlflow.sklearn.log_model()` to understand the explicit logging process.

In [5]:
with mlflow.start_run(run_name="manual-log-decision-tree") as run:
    # Train a new model with different hyperparameters
    dt_model_v2 = DecisionTreeRegressor(max_depth=15, min_samples_split=10, random_state=42)
    dt_model_v2.fit(X_train, y_train)
    
    # Evaluate
    y_pred_v2 = dt_model_v2.predict(X_test)
    mse_v2 = mean_squared_error(y_test, y_pred_v2)
    r2_v2 = r2_score(y_test, y_pred_v2)
    
    # Manually log the model
    mlflow.sklearn.log_model(dt_model_v2, "decision-tree-model")
    
    # Log metrics manually
    mlflow.log_metric("mse", mse_v2)
    mlflow.log_metric("r2_score", r2_v2)
    
    # Save run ID for later
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"Manual Log Run - MSE: {mse_v2:.4f}, R2: {r2_v2:.4f}")

2026/03/14 22:03:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 22:03:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run ID: 0a72566681c94858921e94a39595627e
Manual Log Run - MSE: 0.4314, R2: 0.6708


## 4. Loading a Saved Model

Load the model we just saved using `mlflow.sklearn.load_model()` and verify predictions match.

In [6]:
# Load the model using the run ID
model_uri = f"runs:/{run_id}/decision-tree-model"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Predict with loaded model
y_pred_loaded = loaded_model.predict(X_test)

# Verify predictions match
predictions_match = np.allclose(y_pred_v2, y_pred_loaded)
print(f"Predictions match original model: {predictions_match}")
print(f"Sample predictions (first 5): {y_pred_loaded[:5]}")

Predictions match original model: True
Sample predictions (first 5): [0.51791228 1.203      5.00001    2.47075    2.17416667]


## 5. View Results in MLflow UI

Run the following command in your terminal (with your virtual environment activated):

```bash
mlflow ui --port 5001
```

Then open http://127.0.0.1:5001 in your browser to view the logged experiments.

In [7]:
# Alternatively, try running mlflow ui from within the notebook
# If this doesn't work, use the terminal command above
# !mlflow ui --port 5001